# 03 - n8n + OCI Generative AI using an OpenAI-Compatible Gateway

This notebook follows the written guide: `../N8N-GenAIOCI-Connection.md`.

The goal is to show how a low-code tool such as n8n can call OCI Generative AI through an OpenAI-style API:

```text
n8n OpenAI node
    -> local OpenAI-compatible gateway
    -> OCI Generative AI
    -> OpenAI-style response back to n8n
```

The written guide assumes a gateway application exists. This repo includes a small local gateway at `workshop_gateway/app.py` so the example is runnable.

Official references:

- OCI Generative AI OpenAI-compatible API: https://docs.oracle.com/en-us/iaas/Content/generative-ai/openai-compatible-api.htm
- OCI Generative AI setup and authentication: https://docs.oracle.com/en-us/iaas/Content/generative-ai/setup-oci-api-auth.htm
- n8n OpenAI node documentation: https://docs.n8n.io/integrations/builtin/app-nodes/n8n-nodes-langchain.openai/


Copyright (c) 2026 Oracle and/or its affiliates. Licensed under the Universal Permissive License (UPL), Version 1.0.


## What This Notebook Does

This notebook has two jobs:

1. Prove the local gateway can receive OpenAI-style requests and return OpenAI-style responses.
2. Show exactly how to point n8n at that gateway.

The n8n workflow itself is built in the n8n UI. The notebook acts as the technical runbook and validation tool.


## What n8n Is

n8n is a workflow automation tool. Instead of writing a full application, you build a workflow from connected steps called nodes.

A simple way to think about it:

```text
trigger -> get data -> call a service -> transform the result -> send it somewhere
```

Common n8n patterns are:

| Pattern | Plain meaning | Example in this notebook |
|---|---|---|
| Trigger | Something starts the workflow | Manual run or webhook |
| App node | Connect to a specific app or service | OpenAI-style AI node |
| HTTP Request node | Call any HTTP API directly | `POST /v1/chat/completions` |
| Data mapping | Pass values from one node to another | Use text from one step as the prompt |
| AI step | Send text to a model and use the response | Summarize or answer with OCI Generative AI |

n8n does not need to know the OCI SDK details. In this notebook, n8n sends an OpenAI-style HTTP request to a local gateway. The gateway receives the request, calls OCI Generative AI with OCI authentication, and returns an OpenAI-style response that n8n can use.

```text
n8n workflow
    -> HTTP/OpenAI-style request
    -> local gateway
    -> OCI Generative AI
    -> response back to n8n
```

Important point for this demo: n8n is the workflow builder. The gateway is the translator. OCI Generative AI is still the model service doing the chat work.

This notebook uses the HTTP Request node pattern because it is explicit and easy to inspect. Some n8n setups can also use OpenAI-compatible AI nodes or credentials when a custom base URL is available, but the HTTP Request node keeps the integration clear and stable for the workshop.

Official docs behind this section:

- n8n workflows: https://docs.n8n.io/workflows/
- n8n nodes: https://docs.n8n.io/integrations/builtin/core-nodes/
- n8n HTTP Request node: https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/
- n8n OpenAI node: https://docs.n8n.io/integrations/builtin/app-nodes/n8n-nodes-langchain.openai/
- OCI Generative AI OpenAI-compatible API: https://docs.oracle.com/en-us/iaas/Content/generative-ai/openai-compatible-api.htm


## Install Python Packages

Install dependencies once from the `files` directory by running `uv sync` before opening this notebook.


In [ ]:
# Dependencies are managed by uv. Run `uv sync` from the files directory before opening this notebook.

## Step 1 - Load Workshop Setup

This uses the same `.env` and OCI API-key authentication setup from notebooks `00`, `01`, and `02`.


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
import json
import os
import sys

# Make the repo-level helper package importable from this notebook.
for directory in [Path.cwd(), *Path.cwd().parents]:
    if (directory / "workshop_helpers").exists():
        sys.path.insert(0, str(directory))
        repo_root = directory
        break
else:
    raise FileNotFoundError("Could not find the workshop_helpers folder.")

from workshop_helpers.oci_genai_helpers import load_workshop_env, mask

# Load OCI, model, and gateway settings from .env.
env_path = load_workshop_env()
gateway_host = os.getenv("GATEWAY_HOST", "localhost")
gateway_port = int(os.getenv("GATEWAY_PORT", "8088"))
gateway_base_url = f"http://{gateway_host}:{gateway_port}"

display(Markdown(f"""
**Workshop Setup Loaded**

| Setting | Value |
|---|---|
| `.env` file | `{env_path}` |
| Workshop files directory | `{repo_root}` |
| Gateway base URL | `{gateway_base_url}` |
| n8n OpenAI Base URL | `{gateway_base_url}/v1` |
| Chat model | `{os.getenv('CHAT_MODEL_ID')}` |
| OCI endpoint | `{os.getenv('GENAI_ENDPOINT')}` |
| Compartment | `{mask(os.getenv('OCI_COMPARTMENT_ID'), 32)}` |
"""))

## Step 2 - Inspect The Gateway App

The gateway is a small FastAPI app. It exposes the endpoints n8n needs for this example:

| Endpoint | Purpose |
|---|---|
| `GET /health` | quick gateway and OCI configuration check |
| `GET /v1/models` | OpenAI-style model listing |
| `POST /v1/chat/completions` | OpenAI-style chat completion request |

The gateway ignores the OpenAI API key because OCI authentication happens through your local OCI config file.


In [ ]:
from workshop_gateway.app import app

# Show the routes that the local gateway exposes.
routes = [
    (sorted(route.methods), route.path)
    for route in app.routes
    if hasattr(route, "methods")
]

route_rows = "\n".join(
    f"| `{', '.join(methods)}` | `{path}` |"
    for methods, path in routes
)

display(Markdown(f"""
**Gateway Routes**

| Methods | Path |
|---|---|
{route_rows}
"""))

## Step 3 - Test The Gateway Without Starting A Server

For notebook validation, we can call the FastAPI app in-process. This proves the gateway logic works without opening a port.


In [ ]:
from fastapi.testclient import TestClient

# TestClient calls the gateway app directly inside this Python process.
test_client = TestClient(app)

health_response = test_client.get("/health")
models_response = test_client.get("/v1/models")

health_response.raise_for_status()
models_response.raise_for_status()

display(Markdown(f"""
**Gateway Health Check**

```json
{json.dumps(health_response.json(), indent=2)}
```

**Model Listing**

```json
{json.dumps(models_response.json(), indent=2)}
```
"""))

## Step 4 - Send An OpenAI-Style Chat Request

This is the request shape that tools such as n8n expect. The gateway translates it into an OCI Generative AI call and returns an OpenAI-style response.


In [ ]:
# This is intentionally OpenAI-style, not OCI SDK-style.
chat_payload = {
    "model": os.getenv("CHAT_MODEL_ID"),
    "messages": [
        {
            "role": "system",
            "content": "You are a concise assistant for integration examples.",
        },
        {
            "role": "user",
            "content": "In one sentence, explain what this gateway lets n8n do.",
        },
    ],
    "temperature": 0.2,
    "max_tokens": 200,
}

chat_response = test_client.post("/v1/chat/completions", json=chat_payload)
chat_response.raise_for_status()
chat_result = chat_response.json()
assistant_text = chat_result["choices"][0]["message"]["content"]

display(Markdown(f"""
**Request Sent To The Gateway**

```json
{json.dumps(chat_payload, indent=2)}
```

**Assistant Response**

{assistant_text}

**OpenAI-Style Response Shape**

```json
{json.dumps({k: chat_result[k] for k in ['id', 'object', 'created', 'model']}, indent=2)}
```
"""))

## Step 5 - Start The Local Gateway For n8n

The previous cells tested the gateway in-process. n8n needs a real HTTP server.

When you are ready to connect n8n, change `RUN_LOCAL_GATEWAY` to `True` and run the next cell. It starts Uvicorn in the background.

If you already started the gateway in another terminal, leave this as `False`.


In [ ]:
import subprocess
import time

RUN_LOCAL_GATEWAY = False

gateway_process = None
if RUN_LOCAL_GATEWAY:
    # Start the gateway as a real HTTP server for n8n to call.
    gateway_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "workshop_gateway.app:app",
            "--host",
            gateway_host,
            "--port",
            str(gateway_port),
        ],
        cwd=repo_root,
    )
    time.sleep(3)
    status = "started"
else:
    status = "not started by this notebook"

display(Markdown(f"""
**Gateway Server Status**

| Field | Value |
|---|---|
| Status | `{status}` |
| Gateway URL | `{gateway_base_url}` |
| OpenAI-compatible base URL | `{gateway_base_url}/v1` |

To start it manually from the files directory:

```bash
uv run python -m uvicorn workshop_gateway.app:app --host {gateway_host} --port {gateway_port}
```
"""))

## Step 6 - Configure n8n

The most reliable workflow is an HTTP Request node calling the OpenAI-compatible gateway directly. This demonstrates the same integration pattern while avoiding n8n UI differences around AI/OpenAI credentials.

Import this workflow into n8n:

```text
a-getting-started-guide/n8n/oci-genai-gateway-summarizer.workflow.json
```

The imported workflow uses:

| Node | Purpose |
|---|---|
| Webhook | receives input text |
| HTTP Request | posts to `/v1/chat/completions` on the OCI gateway |
| Respond to Webhook | returns the model summary |

The HTTP Request node is configured for Docker/Podman n8n by default:

```text
http://host.docker.internal:8088/v1/chat/completions
```

If n8n runs directly on your machine instead of Docker/Podman, change that URL to:

```text
http://localhost:8088/v1/chat/completions
```

The `Authorization: Bearer not-used` header is included only because OpenAI-compatible tools often expect it. The gateway ignores it and authenticates to OCI using your local `~/.oci/config` profile.


## Step 7 - Run The Imported Workflow

Start the gateway first:

```bash
uv run python -m uvicorn workshop_gateway.app:app --host localhost --port 8088
```

Then in n8n:

1. Import `oci-genai-gateway-summarizer.workflow.json`.
2. Open the workflow.
3. Click **Listen for test event** on the Webhook node.
4. Send the sample request below.
5. Inspect the HTTP Request node output and the webhook response.

Sample input is also saved here:

```text
a-getting-started-guide/n8n/sample-summary-request.json
```

```json
{
  "text": "Oracle Database can store vectors for semantic search. OCI Generative AI can create embeddings and generate grounded answers. n8n can orchestrate low-code workflows by calling an OpenAI-compatible gateway."
}
```

This is a simpler first example than email because it has no SMTP or mailbox setup. Email, Slack, Teams, or CRM actions can be added afterward as downstream nodes.


## Step 8 - Equivalent `curl` Test

Once the gateway server is running, this request should behave like the in-process test above.

```bash
curl -X POST http://localhost:8088/v1/chat/completions   -H "Content-Type: application/json"   -H "Authorization: Bearer not-used"   -d '{
    "model": "google.gemini-2.5-flash",
    "messages": [
      {"role": "system", "content": "You are a concise summarizer."},
      {"role": "user", "content": "OCI Generative AI powers enterprise AI applications."}
    ],
    "temperature": 0.2,
    "max_tokens": 200
  }'
```


## Troubleshooting

| Symptom | Likely cause | What to check |
|---|---|---|
| n8n cannot connect | Gateway server is not running | Start Uvicorn and open `http://localhost:8088/health` |
| Docker n8n cannot reach gateway | `localhost` points to the container, not your host | Use `http://host.docker.internal:8088/v1` |
| OCI auth error | OCI config profile is missing or invalid | Run notebook `00`; check `OCI_PROFILE` and `OCI_CONFIG_FILE` |
| Model error | n8n requested a model different from the configured gateway model | Use the same value as `CHAT_MODEL_ID` |
| Empty Gemini output | Completion budget used for reasoning | Increase `CHAT_MAX_COMPLETION_TOKENS` |
| Streaming error | Workshop gateway does not implement streaming | Disable streaming in the n8n/OpenAI node if exposed |

To stop a gateway process started by the notebook:

```python
if gateway_process:
    gateway_process.terminate()
```
